In [ ]:
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import AutoModelForSequenceClassification

### Exercise 1

In [ ]:
dataset = load_dataset("rotten_tomatoes")
print("Dataset loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Dataset loaded successfully!


### Exercise 2

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


### Exercise 3

In [ ]:
sample = dataset["train"][0]

print("Sample from training set:")
print(sample)

Sample from training set:
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


### Exercise 4

In [ ]:
print("Feature names:")
print(dataset["train"].features)

Feature names:
{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


### Exercise 5

In [ ]:
num_train_samples = len(dataset["train"])
print(f"Total training samples: {num_train_samples}")

Total training samples: 8530


### Exercise 6

In [ ]:
def preprocess(example):
    example["text_length"] = len(example["text"])
    return example

dataset = dataset.map(preprocess)

print("Updated dataset structure:")
print(dataset)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Updated dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'text_length'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'text_length'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'text_length'],
        num_rows: 1066
    })
})


### Exercise 7

In [ ]:
def preprocess_v2(example):
    example["text_length"] = len(example["text"])
    example["word_count"]  = len(example["text"].split())
    return example

dataset = dataset.map(preprocess_v2)

print("Comparison of text_length vs word_count for first 3 samples:\n")
for i in range(3):
    sample = dataset["train"][i]
    print(f"Sample {i+1}")
    print(f"  Text       : {sample['text'][:80]}...")
    print(f"  Text Length: {sample['text_length']} characters")
    print(f"  Word Count : {sample['word_count']} words")

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Comparison of text_length vs word_count for first 3 samples:

Sample 1
  Text       : the rock is destined to be the 21st century's new " conan " and that he's going ...
  Text Length: 177 characters
  Word Count : 34 words

Sample 2
  Text       : the gorgeously elaborate continuation of " the lord of the rings " trilogy is so...
  Text Length: 226 characters
  Word Count : 39 words

Sample 3
  Text       : effective but too-tepid biopic...
  Text Length: 30 characters
  Word Count : 4 words



### Exercise 8

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

print("Tokenization complete!")
print("\nTokenized dataset structure:")
print(tokenized_datasets)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Tokenization complete!

Tokenized dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'text_length', 'word_count', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'text_length', 'word_count', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'text_length', 'word_count', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
})


In [ ]:
sample = tokenized_datasets["train"][0]

print("=== Tokenized Sample (first 20 values shown) ===\n")
print(f"input_ids      (first 20): {sample['input_ids'][:20]}")
print(f"attention_mask (first 20): {sample['attention_mask'][:20]}")
print(f"\ninput_ids length  : {len(sample['input_ids'])}")
print(f"attention_mask length: {len(sample['attention_mask'])}")

=== Tokenized Sample (first 20 values shown) ===

input_ids      (first 20): [101, 1996, 2600, 2003, 16036, 2000, 2022, 1996, 7398, 2301, 1005, 1055, 2047, 1000, 16608, 1000, 1998, 2008, 2002, 1005]
attention_mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

input_ids length  : 512
attention_mask length: 512


- input_ids -> the text converted to numbers the model understands
- attention_mask -> 1 = real token, 0 = padding (model ignores zeros)

### Exercise 9

In [ ]:
columns_to_remove = ["text", "text_length", "word_count"]
tokenized_datasets = tokenized_datasets.remove_columns(columns_to_remove)

tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

print("Dataset is ready for training!")
print("\nFinal dataset structure:")
print(tokenized_datasets)

Dataset is ready for training!

Final dataset structure:
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
})


### Exercise 10


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print(model)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


### Exercise 11

In [ ]:
training_args = TrainingArguments(
    output_dir="results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
)

print("Training arguments configured!")

Training arguments configured!


In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics
)

print("Trainer is ready!")

Trainer is ready!


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.345856,0.363343,0.840525
2,0.253239,0.368409,0.850844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1068, training_loss=0.32972253753004893, metrics={'train_runtime': 950.9719, 'train_samples_per_second': 17.94, 'train_steps_per_second': 1.123, 'total_flos': 2259893821071360.0, 'train_loss': 0.32972253753004893, 'epoch': 2.0})

### Exercise 12

In [ ]:
results = trainer.evaluate(tokenized_datasets["test"])

print("\n=== Model Evaluation Results ===")
for key, value in results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")


=== Model Evaluation Results ===
  eval_loss: 0.4007
  eval_accuracy: 0.8311
  eval_runtime: 20.4531
  eval_samples_per_second: 52.1190
  eval_steps_per_second: 3.2760
  epoch: 2.0000


In [ ]:
accuracy = results.get("eval_accuracy", None)
if accuracy:
    print(f"\nModel correctly classified {accuracy * 100:.2f}% of test reviews.")


Model correctly classified 83.11% of test reviews.
